In [0]:
# Install required packages
# --no-deps: pandas and numpy are already provided by Databricks Runtime - do not reinstall them
%pip install --force-reinstall --no-deps --no-cache-dir /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/script/databricks_toolkit-2.0.2-py3-none-any.whl
# openpyxl is NOT bundled in Databricks Runtime - must be installed explicitly
%pip install openpyxl --quiet

Processing /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/script/databricks_toolkit-2.0.2-py3-none-any.whl
  Attempting uninstall: databricks-toolkit
    Found existing installation: databricks-toolkit 2.0.2
    Uninstalling databricks-toolkit-2.0.2:
      Successfully uninstalled databricks-toolkit-2.0.2
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python to load the newly installed package
dbutils.library.restartPython()

## Configuration

**IMPORTANT:** Update the paths below to match your environment.

In [0]:
import os
import shutil
from datetime import datetime

# Import analysis components
from databricks_toolkit.analysis import (
    ClusterUtilizationAnalyzer,
    QueryUtilizationAnalyzer,
    SecurityAnalyzer,
    run_consolidated_cluster_analysis
)

# ===================================================================
# CONFIGURATION - Update these paths
# ===================================================================
RULES_PATH      = "/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/script/analyzer_rules.json"
# Discovery output path (from discovery notebook)
DISCOVERY_OUTPUT = "/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery"

# Analysis output directory (where Excel reports will be saved)
ANALYSIS_OUTPUT = "/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis"

# Local temporary directory for analysis processing
# Use a unique timestamp-based directory to avoid permission conflicts
import time
timestamp = int(time.time())
ANALYSIS_TEMP_DIR = f"/tmp/workspace_analysis_{timestamp}"

print("✓ Configuration loaded")
print(f"  Discovery Input: {DISCOVERY_OUTPUT}")
print(f"  Analysis Output: {ANALYSIS_OUTPUT}")
print(f"  Temp Directory: {ANALYSIS_TEMP_DIR}")

✓ Configuration loaded
  Discovery Input: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery
  Analysis Output: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis
  Temp Directory: /tmp/workspace_analysis_1771571142


## Detect Data Paths

Automatically find cluster, query, jobs, and security data.

In [0]:
def find_data_paths(discovery_root):
    """
    Find all data paths from discovery output.
    Handles both workspace-specific folders (ws_*) and direct structure.
    """
    paths = {
        'cluster_input': None,
        'query_input': None,
        'jobs_input': None,
        'security_input': None
    }
    
    try:
        # Convert path to local for os operations
        local_discovery_root = discovery_root.replace('/dbfs/', '/').replace('dbfs:', '')
        if not local_discovery_root.startswith('/') and not local_discovery_root.startswith('/Volumes/'):
            local_discovery_root = '/dbfs' + local_discovery_root
            
        if os.path.exists(local_discovery_root):
            workspace_folders = [d for d in os.listdir(local_discovery_root) 
                               if os.path.isdir(os.path.join(local_discovery_root, d)) and d.startswith('ws_')]
            
            if workspace_folders:
                # Workspace folder structure
                workspace_folder = os.path.join(local_discovery_root, workspace_folders[0])
                paths['cluster_input'] = os.path.join(workspace_folder, 'utilization', 'cluster')
                paths['query_input'] = os.path.join(workspace_folder, 'utilization', 'query')
                paths['jobs_input'] = os.path.join(workspace_folder, 'jobs', 'json')
                
                # Check for security data
                for folder_type in ['json', 'csv']:
                    security_path = os.path.join(workspace_folder, 'security', folder_type)
                    if os.path.exists(security_path):
                        security_files = [f for f in os.listdir(security_path) 
                                        if f.endswith(('.json', '.csv'))]
                        if security_files:
                            paths['security_input'] = os.path.join(security_path, security_files[0])
                            break
            else:
                # Direct structure
                paths['cluster_input'] = os.path.join(local_discovery_root, 'utilization', 'cluster')
                paths['query_input'] = os.path.join(local_discovery_root, 'utilization', 'query')
                paths['jobs_input'] = os.path.join(local_discovery_root, 'jobs', 'json')
                
                # Check for security data
                for folder_type in ['json', 'csv']:
                    security_path = os.path.join(local_discovery_root, 'security', folder_type)
                    if os.path.exists(security_path):
                        security_files = [f for f in os.listdir(security_path) 
                                        if f.endswith(('.json', '.csv'))]
                        if security_files:
                            paths['security_input'] = os.path.join(security_path, security_files[0])
                            break
    except Exception as e:
        print(f"Error determining paths: {e}")
    
    return paths

# Find data paths
data_paths = find_data_paths(DISCOVERY_OUTPUT)

print("\nData paths identified:")
print(f"  📊 Cluster: {data_paths['cluster_input']}")
print(f"     Exists: {os.path.exists(data_paths['cluster_input']) if data_paths['cluster_input'] else False}")

print(f"  📊 Query: {data_paths['query_input']}")
print(f"     Exists: {os.path.exists(data_paths['query_input']) if data_paths['query_input'] else False}")

print(f"  📊 Jobs: {data_paths['jobs_input']}")
print(f"     Exists: {os.path.exists(data_paths['jobs_input']) if data_paths['jobs_input'] else False}")

print(f"  📊 Security: {data_paths['security_input']}")
print(f"     Exists: {os.path.exists(data_paths['security_input']) if data_paths['security_input'] else False}")

# Verify at least cluster data exists
if not data_paths['cluster_input'] or not os.path.exists(data_paths['cluster_input']):
    print("\n❌ ERROR: Cluster data not found!")
    print("   Please verify discovery notebook ran successfully with scan_type='deep'")
    dbutils.notebook.exit("Discovery data not found")


Data paths identified:
  📊 Cluster: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/utilization/cluster
     Exists: True
  📊 Query: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/utilization/query
     Exists: True
  📊 Jobs: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/jobs/json
     Exists: True
  📊 Security: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/security/json/adb-3484933024933655_security_20260220_070135.json
     Exists: True


## Run Complete Analysis

This cell executes ALL analyzers and generates comprehensive reports:
1. **Cluster Analysis** - 7 consolidated analyzers (utilization, network, RI, worker nodes, single-node, autoscaling, recommendations)
2. **Job Analysis** - Job configuration and execution patterns
3. **Query Analysis** - SQL Warehouse query performance (if data available)
4. **Security Analysis** - Comprehensive security posture assessment

All reports are saved as Excel files in the output directory.

In [0]:
print("="*80)
print("RUNNING COMPREHENSIVE WORKSPACE ANALYSIS")
print("="*80)
print(f"\nThis will generate:")
print(f"  1. Cluster Analysis (7 analyzers consolidated)")
print(f"  2. Job Analysis")
print(f"  3. Query Analysis (if data available)")
print(f"  4. Security Analysis (if data available)")
print(f"\nOutput: {ANALYSIS_OUTPUT}")
print("="*80 + "\n")

# Create temporary directory with proper cleanup
temp_dir = os.path.join(ANALYSIS_TEMP_DIR, 'comprehensive')

# Clean up any existing temp directory first
if os.path.exists(ANALYSIS_TEMP_DIR):
    try:
        shutil.rmtree(ANALYSIS_TEMP_DIR)
        print(f"✓ Cleaned up existing temp directory")
    except Exception as e:
        print(f"⚠️  Warning: Could not clean existing temp dir: {e}")

# Create fresh temp directory
try:
    os.makedirs(temp_dir, mode=0o777, exist_ok=True)
    print(f"✓ Created temp directory: {temp_dir}")
except Exception as e:
    print(f"❌ Error creating temp directory: {e}")
    # Fallback to a simpler path
    temp_dir = f"/tmp/analysis_{timestamp}"
    os.makedirs(temp_dir, mode=0o777, exist_ok=True)
    print(f"✓ Using fallback temp directory: {temp_dir}")

# Determine output path based on storage type
if ANALYSIS_OUTPUT.startswith('/Volumes/'):
    output_path = ANALYSIS_OUTPUT
elif ANALYSIS_OUTPUT.startswith('/dbfs/') or ANALYSIS_OUTPUT.startswith('dbfs:/'):
    output_path = ANALYSIS_OUTPUT.replace('dbfs:/', '/dbfs/')
    if not output_path.startswith('/dbfs'):
        output_path = '/dbfs' + output_path
else:
    output_path = '/dbfs' + ANALYSIS_OUTPUT

os.makedirs(output_path, exist_ok=True)

reports_generated = []

try:
    # =========================================================================
    # 1. CLUSTER + JOB ANALYSIS
    # =========================================================================
    print(f"\n{'='*80}")
    print("STEP 1/4: RUNNING CLUSTER + JOB ANALYSIS")
    print(f"{'='*80}\n")
    
    cluster_results = run_consolidated_cluster_analysis(
        cluster_input_path=data_paths['cluster_input'],
        output_dir=temp_dir,
        job_input_path=data_paths['jobs_input'],
        rules_path=RULES_PATH
    )
    
    # Copy cluster report
    if cluster_results.get('cluster_output_file') and os.path.exists(cluster_results['cluster_output_file']):
        cluster_dst = os.path.join(output_path, os.path.basename(cluster_results['cluster_output_file']))
        shutil.copy2(cluster_results['cluster_output_file'], cluster_dst)
        reports_generated.append(f"Cluster Analysis: {os.path.basename(cluster_results['cluster_output_file'])}")
        print(f"\n✓ Cluster report saved: {os.path.basename(cluster_results['cluster_output_file'])}")
    
    # Copy job report
    if cluster_results.get('job_output_file') and os.path.exists(cluster_results['job_output_file']):
        job_dst = os.path.join(output_path, os.path.basename(cluster_results['job_output_file']))
        shutil.copy2(cluster_results['job_output_file'], job_dst)
        reports_generated.append(f"Job Analysis: {os.path.basename(cluster_results['job_output_file'])}")
        print(f"✓ Job report saved: {os.path.basename(cluster_results['job_output_file'])}")
    
    # =========================================================================
    # 2. QUERY ANALYSIS
    # =========================================================================
    print(f"\n{'='*80}")
    print("STEP 2/4: RUNNING QUERY ANALYSIS")
    print(f"{'='*80}\n")
    
    if data_paths['query_input'] and os.path.exists(data_paths['query_input']):
        try:
            query_analyzer = QueryUtilizationAnalyzer(
                input_path=data_paths['query_input'],
                output_dir=temp_dir
            )
            
            query_results = query_analyzer.analyze()
            
            # Copy query report
            if query_results.get('output_file') and os.path.exists(query_results['output_file']):
                query_dst = os.path.join(output_path, os.path.basename(query_results['output_file']))
                shutil.copy2(query_results['output_file'], query_dst)
                reports_generated.append(f"Query Analysis: {os.path.basename(query_results['output_file'])}")
                print(f"\n✓ Query report saved: {os.path.basename(query_results['output_file'])}")
        except FileNotFoundError as e:
            print(f"⚠️  Query data not found: {e}")
            print("   This is normal if no SQL Warehouses were active during discovery")
        except Exception as e:
            print(f"⚠️  Query analysis skipped: {e}")
    else:
        print("⚠️  Query data not found - skipping query analysis")
        print("   This is normal if no SQL Warehouses were active during discovery")
    
    # =========================================================================
    # 3. SECURITY ANALYSIS
    # =========================================================================
    print(f"\n{'='*80}")
    print("STEP 3/4: RUNNING SECURITY ANALYSIS")
    print(f"{'='*80}\n")
    
    if data_paths['security_input'] and os.path.exists(data_paths['security_input']):
        try:
            print(f"✓ Security data located: {data_paths['security_input']}")
            
            security_analyzer = SecurityAnalyzer(
                discovery_path=data_paths['security_input'],
                output_dir=temp_dir
            )
            
            security_results = security_analyzer.analyze()
            
            # Copy security report - handle different return structures
            output_file = None
            
            # Debug: print what we got back
            print(f"\nDebug - security_results keys: {security_results.keys()}")
            
            # Try different keys and structures
            if 'output_files' in security_results:
                result_data = security_results['output_files']
                if isinstance(result_data, dict):
                    # If it's a dict, try to extract the file path
                    output_file = result_data.get('xlsx') or result_data.get('excel') or result_data.get('file')
                elif isinstance(result_data, str):
                    output_file = result_data
            elif 'output_file' in security_results:
                output_file = security_results['output_file']
            
            # If still no file, try to find any xlsx file in temp_dir
            if not output_file:
                print("Debug - Searching temp directory for security report...")
                for file in os.listdir(temp_dir):
                    if file.startswith('Security_Analysis') and file.endswith('.xlsx'):
                        output_file = os.path.join(temp_dir, file)
                        print(f"Debug - Found security report: {output_file}")
                        break
            
            if output_file and isinstance(output_file, str) and os.path.exists(output_file):
                security_dst = os.path.join(output_path, os.path.basename(output_file))
                shutil.copy2(output_file, security_dst)
                reports_generated.append(f"Security Analysis: {os.path.basename(output_file)}")
                print(f"\n✓ Security report saved: {os.path.basename(output_file)}")
            else:
                print(f"⚠️  Security report file not found")
                print(f"   Result structure: {security_results}")
        except Exception as e:
            print(f"⚠️  Security analysis failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️  Security data not found - skipping security analysis")
        print("   Run discovery with scan_type='deep' to collect security data")
    
    # =========================================================================
    # SUMMARY
    # =========================================================================
    print(f"\n{'='*80}")
    print("✓ COMPREHENSIVE ANALYSIS COMPLETE")
    print(f"{'='*80}\n")
    
    print(f"Generated {len(reports_generated)} report(s):\n")
    for report in reports_generated:
        print(f"  📊 {report}")
    
    print(f"\nAll reports saved to: {ANALYSIS_OUTPUT}")
    print(f"\n{'='*80}")
    
except Exception as e:
    print(f"\n❌ Error during comprehensive analysis: {str(e)}")
    import traceback
    traceback.print_exc()
    
finally:
    # Clean up temporary files
    try:
        if os.path.exists(ANALYSIS_TEMP_DIR):
            shutil.rmtree(ANALYSIS_TEMP_DIR)
            print(f"\n✓ Cleaned up temporary files")
        elif os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
            print(f"\n✓ Cleaned up temporary files")
    except Exception as e:
        print(f"⚠️  Warning: Could not clean up temp files: {e}")

RUNNING COMPREHENSIVE WORKSPACE ANALYSIS

This will generate:
  1. Cluster Analysis (7 analyzers consolidated)
  2. Job Analysis
  3. Query Analysis (if data available)
  4. Security Analysis (if data available)

Output: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis

✓ Created temp directory: /tmp/workspace_analysis_1771571142/comprehensive

STEP 1/4: RUNNING CLUSTER + JOB ANALYSIS

CONSOLIDATED CLUSTER ANALYSIS

Started at: 2026-02-20 07:05:55
Input: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/utilization/cluster
Output: /tmp/workspace_analysis_1771571142/comprehensive/Cluster_Analysis_Consolidated_20260220_070555.xlsx

[1/7] Running Cluster Utilization Analyzer...
2026-02-20 07:05:55,305 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-02-20 07:05:55,310 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
⚠️  Warning: analyzer_rules.json not loaded. Using built-in default t

Traceback (most recent call last):
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-fda2ba7b-7f8c-4dc4-a1c6-aff2ce1c638d/lib/python3.12/site-packages/databricks_toolkit/analysis/analyzer.py", line 1215, in run_consolidated_cluster_analysis
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.12/site-packages/pandas/io/excel/_openpyxl.py", line 57, in __init__
    from openpyxl.workbook import Workbook
ModuleNotFoundError: No module named 'openpyxl'


✓ Security data located: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/discovery/security/json/adb-3484933024933655_security_20260220_070135.json
2026-02-20 07:05:56,219 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-02-20 07:05:56,219 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
✓ Loaded security checks metadata from: security_checks_metadata.json

SECURITY ANALYSIS - Databricks Workspace Security Assessment

📥 Loading workspace configuration...
  ✓ Loaded security data from discovery
✓ Loaded configuration for workspace: adb-3484933024933655
  Cloud Type: Azure
  Security data available: True

Analyzing Network Security...
  ✓ Completed 4 network security checks

Analyzing Identity & Access Management...
  ✓ Completed 4 identity & access checks

Analyzing Data Protection...
  ✓ Completed 4 data protection checks

Analyzing Governance & Compliance...
  ✓ Completed 4 governance checks

Ana

Traceback (most recent call last):
  File "/home/spark-fda2ba7b-7f8c-4dc4-a1c6-af/.ipykernel/13544/command-8822644797312830-2305301182", line 123, in <module>
    security_results = security_analyzer.analyze()
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-fda2ba7b-7f8c-4dc4-a1c6-aff2ce1c638d/lib/python3.12/site-packages/databricks_toolkit/analysis/security_analyzer.py", line 224, in analyze
    output_files = self._save_to_excel(output_prefix)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/local_disk0/.ephemeral_nfs/envs/pythonEnv-fda2ba7b-7f8c-4dc4-a1c6-aff2ce1c638d/lib/python3.12/site-packages/databricks_toolkit/analysis/security_analyzer.py", line 1318, in _save_to_excel
    with pd.ExcelWriter(temp_excel_path, engine='openpyxl') as writer:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.12/site-packages/pandas/io/excel/_openpyxl.py", line 57, in __init__
    fr

## View Generated Reports

List all Excel reports generated by the analysis.

In [0]:
print("="*80)
print("ANALYSIS REPORTS GENERATED")
print("="*80)
print(f"\nOutput directory: {ANALYSIS_OUTPUT}\n")

try:
    files = dbutils.fs.ls(ANALYSIS_OUTPUT)
    
    excel_files = [f for f in files if f.name.endswith('.xlsx')]
    
    if excel_files:
        print(f"Found {len(excel_files)} Excel report(s):\n")
        for file_info in sorted(excel_files, key=lambda x: x.name):
            size_mb = file_info.size / (1024*1024)
            print(f"  📊 {file_info.name}")
            print(f"     Size: {size_mb:.2f} MB")
            print(f"     Path: {file_info.path}\n")
    else:
        print("⚠️  No Excel reports found")
        
except Exception as e:
    print(f"Error listing reports: {e}")

print("="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)
print("\n📥 Download the Excel reports to view detailed analysis and recommendations.")

ANALYSIS REPORTS GENERATED

Output directory: /Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis

Found 4 Excel report(s):

  📊 Cluster_Analysis_Consolidated_20260205_151813.xlsx
     Size: 0.01 MB
     Path: dbfs:/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis/Cluster_Analysis_Consolidated_20260205_151813.xlsx

  📊 Job_Analysis_adb-3484933024933655_20260205_151815.xlsx
     Size: 0.01 MB
     Path: dbfs:/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis/Job_Analysis_adb-3484933024933655_20260205_151815.xlsx

  📊 SQL_Query_Analysis_20260205_151816.xlsx
     Size: 0.01 MB
     Path: dbfs:/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis/SQL_Query_Analysis_20260205_151816.xlsx

  📊 Security_Analysis_adb-3484933024933655_20260205_151817.xlsx
     Size: 0.01 MB
     Path: dbfs:/Volumes/ext_nyc_cat_ci/etlschema/etlvolume/out/analysis/Security_Analysis_adb-3484933024933655_20260205_151817.xlsx

✓ ANALYSIS COMPLETE

📥 Download the Excel reports to view detailed ana